<a href="https://colab.research.google.com/github/Fahad-Hafeez/phishing-ml-classifier-comparison/blob/main/phishing_benchmark_optimised.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Cell 0 -- Installs & Imports

In [ ]:
# Install all dependencies (run once per session)
!pip install pandas statsmodels imbalanced-learn xgboost shap --quiet

import gc
import warnings
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns
from scipy import stats
from scipy.stats import spearmanr

from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import (
    train_test_split, GridSearchCV, RandomizedSearchCV,
    StratifiedKFold, cross_val_score
)
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report, RocCurveDisplay
)
from imblearn.over_sampling import SMOTE
from statsmodels.stats.contingency_tables import mcnemar
import shap

warnings.filterwarnings('ignore')
shap.initjs()

SEED = 42
np.random.seed(SEED)
print("All imports successful.")

## Cell 1 -- Load Dataset

In [ ]:
df = pd.read_csv('uci-ml-phishing-dataset.csv')
print("Shape:", df.shape)
print(df.head(3))
print("\nDtypes:\n", df.dtypes)

## Cell 2 -- Exploratory Data Analysis

In [ ]:
print("Missing values:", df.isnull().sum().sum())
print("\nClass distribution:")
total = len(df)
for cls, cnt in df['Result'].value_counts().items():
    label = "Phishing" if cls == 1 else "Legitimate"
    print(f"  {label} ({cls}): {cnt}  ({cnt / total * 100:.1f}%)")
print("\nNote: 56:44 ratio is near-balanced -- no severe class imbalance.")

## Cell 3 -- Figure 1: Class Distribution

In [ ]:
class_counts = df['Result'].value_counts().sort_index(ascending=False)
labels = ['Phishing', 'Legitimate']
colors = ['#E74C3C', '#2E86C1']

plt.figure(figsize=(7, 5))
bars = plt.bar(labels, class_counts.values, color=colors)
for bar, v in zip(bars, class_counts.values):
    plt.text(bar.get_x() + bar.get_width() / 2,
             v + 40, str(v), ha='center', va='bottom', fontsize=11)

plt.xlabel('Class')
plt.ylabel('Number of Instances')
plt.title('Class Distribution of Phishing Websites Dataset')
plt.tight_layout()
plt.savefig('fig1_class_distribution.pdf', dpi=300, bbox_inches='tight')
plt.show()
print("Figure 1 saved.")

## Cell 4 -- Preprocessing: Train-Test Split, Scaling, SMOTE

**Reviewer 1 note (pipeline clarity):**
Two evaluation streams run in parallel after this split:
- **Stream 1**: The held-out test set (20%) is used *only* for McNemar test and Table 1 metrics. It is never seen during hyperparameter tuning or cross-validation.
- **Stream 2**: 10-fold CV is conducted *exclusively on the training partition* (80%) to compute Cohen's d and bootstrap CIs.

In [ ]:
# ---- Features and target ----
X = df.drop('Result', axis=1)
y = df['Result']
feature_names = list(X.columns)

# ---- Stratified 80/20 split (LOCKED test set) ----
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=SEED, stratify=y
)
print(f"X_train: {X_train.shape} | X_test: {X_test.shape}")
print(f"Train class dist: {dict(y_train.value_counts())}")
print(f"Test  class dist: {dict(y_test.value_counts())}")

# ---- StandardScaler fitted on training data ONLY (no leakage) ----
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)
print("\nScaler fitted on X_train only. Applied transform-only to X_test.")

# ---- SMOTE applied to training partition only ----
sm = SMOTE(random_state=SEED)
X_train_balanced, y_train_balanced = sm.fit_resample(X_train_scaled, y_train)
print(f"\nSMOTE training set: {X_train_balanced.shape}")
print(f"SMOTE class dist: {dict(pd.Series(y_train_balanced).value_counts())}")
print("Test set remains untouched (reflects real-world distribution).")

# ---- XGBoost label encoding: -1/1 -> 0/1 ----
le = LabelEncoder()
y_train_xgb     = le.fit_transform(y_train)
y_test_xgb      = le.transform(y_test)
y_train_bal_xgb = le.transform(y_train_balanced)
print("\nXGBoost label encoding: -1->0, 1->1")

## Cell 5 -- Helper Functions

In [ ]:
def evaluate_model(model, X, y):
    """Evaluate LR / RF / SVM (have predict_proba)."""
    y_pred = model.predict(X)
    auc = None
    if hasattr(model, 'predict_proba'):
        y_proba = model.predict_proba(X)
        if 1 in model.classes_:
            idx = list(model.classes_).index(1)
            auc = roc_auc_score(y, y_proba[:, idx])
    return {
        'accuracy':  accuracy_score(y, y_pred),
        'precision': precision_score(y, y_pred, average='macro'),
        'recall':    recall_score(y, y_pred, average='macro'),
        'f1_score':  f1_score(y, y_pred, average='macro'),
        'auc_roc':   auc
    }


def evaluate_xgb(model, X, y_bin, le, y_orig):
    """Evaluate XGBoost: handles 0/1 -> -1/1 label remapping."""
    y_pred_bin = model.predict(X)
    y_pred     = le.inverse_transform(y_pred_bin)
    y_proba    = model.predict_proba(X)[:, 1]
    return {
        'accuracy':  accuracy_score(y_orig, y_pred),
        'precision': precision_score(y_orig, y_pred, average='macro'),
        'recall':    recall_score(y_orig, y_pred, average='macro'),
        'f1_score':  f1_score(y_orig, y_pred, average='macro'),
        'auc_roc':   roc_auc_score(y_bin, y_proba)
    }


def cohens_d(a, b):
    """Cohen's d with near-zero variance guard."""
    std_a = np.std(a, ddof=1)
    std_b = np.std(b, ddof=1)
    pooled = np.sqrt((std_a ** 2 + std_b ** 2) / 2)
    if pooled < 1e-9:
        return 0.0 if np.isclose(np.mean(a), np.mean(b)) else (
            np.inf if np.mean(a) > np.mean(b) else -np.inf
        )
    return (np.mean(a) - np.mean(b)) / pooled


def bootstrap_ci(a, b, n=5000, ci=0.95):
    """Bootstrap 95% CI for mean F1 difference (a - b)."""
    diffs = a - b
    bs_means = [
        np.mean(np.random.choice(diffs, len(diffs), replace=True))
        for _ in range(n)
    ]
    lo = np.percentile(bs_means, (1 - ci) / 2 * 100)
    hi = np.percentile(bs_means, (1 + ci) / 2 * 100)
    return lo, hi


print("Helper functions defined: evaluate_model, evaluate_xgb, cohens_d, bootstrap_ci")

## Cell 6 -- Logistic Regression Tuning (Both Conditions)

In [ ]:
lr_param_grid = [
    {
        'penalty': ['l1'],
        'solver':  ['liblinear', 'saga'],
        'C':       [0.001, 0.01, 0.1, 1, 10, 100]
    },
    {
        'penalty': ['l2'],
        'solver':  ['newton-cg', 'lbfgs', 'liblinear', 'sag', 'saga'],
        'C':       [0.001, 0.01, 0.1, 1, 10, 100]
    },
    {
        'penalty': ['elasticnet'],
        'solver':  ['saga'],
        'C':       [0.001, 0.01, 0.1, 1, 10, 100],
        'l1_ratio':[0.1, 0.5, 0.9]
    }
]

# Unbalanced
print("Tuning LR (unbalanced)...")
gs_lr = GridSearchCV(
    LogisticRegression(random_state=SEED, max_iter=1000),
    lr_param_grid, cv=5, scoring='f1_macro', n_jobs=-1
)
gs_lr.fit(X_train_scaled, y_train)
best_lr = gs_lr.best_estimator_
best_lr_params = gs_lr.best_params_
print(f"  Best: {best_lr_params}  |  CV F1: {gs_lr.best_score_:.4f}")
del gs_lr; gc.collect()

# SMOTE-balanced
print("Tuning LR (SMOTE-balanced)...")
gs_lr_bal = GridSearchCV(
    LogisticRegression(random_state=SEED, max_iter=1000),
    lr_param_grid, cv=5, scoring='f1_macro', n_jobs=-1
)
gs_lr_bal.fit(X_train_balanced, y_train_balanced)
best_lr_bal = gs_lr_bal.best_estimator_
best_lr_bal_params = gs_lr_bal.best_params_
print(f"  Best: {best_lr_bal_params}  |  CV F1: {gs_lr_bal.best_score_:.4f}")
del gs_lr_bal; gc.collect()
print("LR tuning complete.")

## Cell 7 -- Random Forest Tuning (Both Conditions)

In [ ]:
# max_depth=None removed: unbounded trees are slow and add little F1 benefit
rf_param_grid = {
    'n_estimators':      [100, 200],
    'max_depth':         [10, 20, 30],
    'min_samples_split': [2, 5]
}

# Unbalanced
print("Tuning RF (unbalanced)...")
gs_rf = GridSearchCV(
    RandomForestClassifier(random_state=SEED),
    rf_param_grid, cv=5, scoring='f1_macro', n_jobs=-1
)
gs_rf.fit(X_train_scaled, y_train)
best_rf = gs_rf.best_estimator_
best_rf_params = gs_rf.best_params_
print(f"  Best: {best_rf_params}  |  CV F1: {gs_rf.best_score_:.4f}")
del gs_rf; gc.collect()

# SMOTE-balanced
print("Tuning RF (SMOTE-balanced)...")
gs_rf_bal = GridSearchCV(
    RandomForestClassifier(random_state=SEED),
    rf_param_grid, cv=5, scoring='f1_macro', n_jobs=-1
)
gs_rf_bal.fit(X_train_balanced, y_train_balanced)
best_rf_bal = gs_rf_bal.best_estimator_
best_rf_bal_params = gs_rf_bal.best_params_
print(f"  Best: {best_rf_bal_params}  |  CV F1: {gs_rf_bal.best_score_:.4f}")
del gs_rf_bal; gc.collect()
print("RF tuning complete.")

## Cell 8 -- SVM Tuning (Optimised -- Expanded Grid, No Probability During Search)

**Key performance fix:** `SVC(probability=True)` inside `GridSearchCV` triggers Platt scaling,
which runs an *internal* 5-fold CV on every single `.fit()` call. With `GridSearchCV(cv=5)`,
that is 5 outer x 5 inner = 25 fits per combination. With ~24 combinations this was causing
runtime crashes.

**Fix:** Search with `probability=False`, then refit the single best model with `probability=True`
exactly once (needed for AUC-ROC and SHAP).

**Reviewer 2 fix (grid expansion):** Old grid `C=[0.1, 1, 10]` had the optimal C=10 at
the boundary. New grid extends to C=1000 so the optimum can fall in the interior.

In [ ]:
# Expanded SVM grid -- separate dicts for linear vs RBF to avoid redundant combos
svm_param_grid = [
    {
        'C':      [0.1, 1, 10, 100, 1000],
        'kernel': ['linear']
    },
    {
        'C':      [0.1, 1, 10, 100, 1000],
        'kernel': ['rbf'],
        'gamma':  ['scale', 'auto', 0.001, 0.01, 0.1]
    }
]
all_C_vals = sorted(set(c for p in svm_param_grid for c in p['C']))


def tune_svm(X_tr, y_tr, label):
    """Tune SVM without probability=True, then refit once with it."""
    gs = GridSearchCV(
        SVC(random_state=SEED, probability=False, cache_size=2000),
        svm_param_grid, cv=5, scoring='f1_macro', n_jobs=-1, verbose=0
    )
    gs.fit(X_tr, y_tr)
    bp = gs.best_params_
    print(f"  SVM ({label}) best: {bp}  |  CV F1: {gs.best_score_:.4f}")

    # Boundary check
    if bp['C'] in [min(all_C_vals), max(all_C_vals)]:
        print(f"  WARNING: C={bp['C']} is at grid boundary -- consider expanding further.")
    else:
        print(f"  OK: C={bp['C']} is interior to search range.")

    # Refit ONCE with probability=True (for AUC-ROC and SHAP)
    model = SVC(**bp, random_state=SEED, probability=True, cache_size=2000)
    model.fit(X_tr, y_tr)
    del gs; gc.collect()
    return model, bp


print("Tuning SVM (unbalanced)...")
best_svm, best_svm_params = tune_svm(X_train_scaled, y_train, 'unbalanced')

print("Tuning SVM (SMOTE-balanced)...")
best_svm_bal, best_svm_bal_params = tune_svm(X_train_balanced, y_train_balanced, 'SMOTE')

print("SVM tuning complete.")

## Cell 9 -- XGBoost Tuning (RandomizedSearchCV -- Faster than Full Grid)

In [ ]:
# RandomizedSearchCV with n_iter=20 instead of GridSearchCV over 72 combos
# Reduces from 720 fits down to ~200 fits while still exploring the full space
xgb_param_dist = {
    'n_estimators':     [100, 200, 300],
    'max_depth':        [3, 6, 9],
    'learning_rate':    [0.05, 0.1, 0.2],
    'subsample':        [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

# Unbalanced
print("Tuning XGBoost (unbalanced)...")
gs_xgb = RandomizedSearchCV(
    XGBClassifier(random_state=SEED, eval_metric='logloss',
                  use_label_encoder=False, verbosity=0),
    xgb_param_dist, n_iter=20, cv=5, scoring='f1_macro',
    n_jobs=-1, random_state=SEED, verbose=0
)
gs_xgb.fit(X_train_scaled, y_train_xgb)
best_xgb = gs_xgb.best_estimator_
best_xgb_params = gs_xgb.best_params_
print(f"  Best: {best_xgb_params}  |  CV F1: {gs_xgb.best_score_:.4f}")
del gs_xgb; gc.collect()

# SMOTE-balanced
print("Tuning XGBoost (SMOTE-balanced)...")
gs_xgb_bal = RandomizedSearchCV(
    XGBClassifier(random_state=SEED, eval_metric='logloss',
                  use_label_encoder=False, verbosity=0),
    xgb_param_dist, n_iter=20, cv=5, scoring='f1_macro',
    n_jobs=-1, random_state=SEED, verbose=0
)
gs_xgb_bal.fit(X_train_balanced, y_train_bal_xgb)
best_xgb_bal = gs_xgb_bal.best_estimator_
best_xgb_bal_params = gs_xgb_bal.best_params_
print(f"  Best: {best_xgb_bal_params}  |  CV F1: {gs_xgb_bal.best_score_:.4f}")
del gs_xgb_bal; gc.collect()
print("XGBoost tuning complete.")

## Cell 10 -- Evaluate All 8 Models on Test Set + Training Set

In [ ]:
# ---- Test set evaluations (Stream 1 -- held-out) ----
lr_eval      = evaluate_model(best_lr,      X_test_scaled, y_test)
rf_eval      = evaluate_model(best_rf,      X_test_scaled, y_test)
svm_eval     = evaluate_model(best_svm,     X_test_scaled, y_test)
xgb_eval     = evaluate_xgb(best_xgb,      X_test_scaled, y_test_xgb, le, y_test)

lr_eval_bal  = evaluate_model(best_lr_bal,  X_test_scaled, y_test)
rf_eval_bal  = evaluate_model(best_rf_bal,  X_test_scaled, y_test)
svm_eval_bal = evaluate_model(best_svm_bal, X_test_scaled, y_test)
xgb_eval_bal = evaluate_xgb(best_xgb_bal,  X_test_scaled, y_test_xgb, le, y_test)

# ---- Training set evaluations (for generalisation gap table) ----
lr_train_eval      = evaluate_model(best_lr,      X_train_scaled,   y_train)
rf_train_eval      = evaluate_model(best_rf,      X_train_scaled,   y_train)
svm_train_eval     = evaluate_model(best_svm,     X_train_scaled,   y_train)
xgb_train_eval     = evaluate_xgb(best_xgb,      X_train_scaled,   y_train_xgb, le, y_train)

lr_train_eval_bal  = evaluate_model(best_lr_bal,  X_train_balanced, y_train_balanced)
rf_train_eval_bal  = evaluate_model(best_rf_bal,  X_train_balanced, y_train_balanced)
svm_train_eval_bal = evaluate_model(best_svm_bal, X_train_balanced, y_train_balanced)
xgb_train_eval_bal = evaluate_xgb(best_xgb_bal,  X_train_balanced, y_train_bal_xgb, le, y_train_balanced)

# ---- Predictions (for McNemar + confusion matrices + error analysis) ----
pred_lr      = best_lr.predict(X_test_scaled)
pred_rf      = best_rf.predict(X_test_scaled)
pred_svm     = best_svm.predict(X_test_scaled)
pred_xgb     = le.inverse_transform(best_xgb.predict(X_test_scaled))

pred_lr_bal  = best_lr_bal.predict(X_test_scaled)
pred_rf_bal  = best_rf_bal.predict(X_test_scaled)
pred_svm_bal = best_svm_bal.predict(X_test_scaled)
pred_xgb_bal = le.inverse_transform(best_xgb_bal.predict(X_test_scaled))

print("All 8 models evaluated. Test and training metrics computed.")

## Cell 11 -- Table 1: Performance Comparison (Test Set)

In [ ]:
results = [
    {'Model': 'Logistic Regression', 'Condition': 'Unbalanced', **lr_eval},
    {'Model': 'Random Forest',       'Condition': 'Unbalanced', **rf_eval},
    {'Model': 'SVM',                 'Condition': 'Unbalanced', **svm_eval},
    {'Model': 'XGBoost',             'Condition': 'Unbalanced', **xgb_eval},
    {'Model': 'Logistic Regression', 'Condition': 'SMOTE',      **lr_eval_bal},
    {'Model': 'Random Forest',       'Condition': 'SMOTE',      **rf_eval_bal},
    {'Model': 'SVM',                 'Condition': 'SMOTE',      **svm_eval_bal},
    {'Model': 'XGBoost',             'Condition': 'SMOTE',      **xgb_eval_bal},
]

results_df = pd.DataFrame(results)
print("=== TABLE 1: Test-set classification performance ===")
print(results_df.to_string(index=False, float_format="%.4f"))

## Cell 12 -- Table 2: Training vs Test Performance (Generalisation Gap)

Requested by Reviewer 2. Small F1_Gap values confirm no data leakage and consistent generalisation
across classifier families. A flag is raised for any model where F1_Gap > 0.02.

In [ ]:
comp_data = [
    {'Model': 'Logistic Regression', 'Condition': 'Unbalanced',
     'Train_F1': lr_train_eval['f1_score'],     'Test_F1': lr_eval['f1_score']},
    {'Model': 'Random Forest',       'Condition': 'Unbalanced',
     'Train_F1': rf_train_eval['f1_score'],     'Test_F1': rf_eval['f1_score']},
    {'Model': 'SVM',                 'Condition': 'Unbalanced',
     'Train_F1': svm_train_eval['f1_score'],    'Test_F1': svm_eval['f1_score']},
    {'Model': 'XGBoost',             'Condition': 'Unbalanced',
     'Train_F1': xgb_train_eval['f1_score'],    'Test_F1': xgb_eval['f1_score']},
    {'Model': 'Logistic Regression', 'Condition': 'SMOTE',
     'Train_F1': lr_train_eval_bal['f1_score'], 'Test_F1': lr_eval_bal['f1_score']},
    {'Model': 'Random Forest',       'Condition': 'SMOTE',
     'Train_F1': rf_train_eval_bal['f1_score'], 'Test_F1': rf_eval_bal['f1_score']},
    {'Model': 'SVM',                 'Condition': 'SMOTE',
     'Train_F1': svm_train_eval_bal['f1_score'],'Test_F1': svm_eval_bal['f1_score']},
    {'Model': 'XGBoost',             'Condition': 'SMOTE',
     'Train_F1': xgb_train_eval_bal['f1_score'],'Test_F1': xgb_eval_bal['f1_score']},
]

comparison_df = pd.DataFrame(comp_data)
comparison_df['F1_Gap'] = comparison_df['Train_F1'] - comparison_df['Test_F1']

print("=== TABLE 2: Generalisation gap (Train F1 - Test F1) ===")
print(comparison_df.to_string(index=False, float_format="%.4f"))

# Overfitting flags
flagged = comparison_df[comparison_df['F1_Gap'] > 0.02]
print()
if flagged.empty:
    print("No models show F1_Gap > 0.02 -- no concerning overfitting detected.")
else:
    for _, row in flagged.iterrows():
        print(f"WARNING: {row['Model']} ({row['Condition']}): F1_Gap = {row['F1_Gap']:.4f}")

# LaTeX export
rows_latex = []
for cond in ['Unbalanced', 'SMOTE']:
    sub = comparison_df[comparison_df['Condition'] == cond]
    best_idx = sub['Test_F1'].idxmax()
    for idx, row in sub.iterrows():
        tf = (f"\\textbf{{{row['Test_F1']:.4f}}}" if idx == best_idx
              else f"{row['Test_F1']:.4f}")
        rows_latex.append(
            f"{row['Model']} & {row['Condition']} & "
            f"{row['Train_F1']:.4f} & {tf} & {row['F1_Gap']:.4f} \\\\"
        )
    if cond == 'Unbalanced':
        rows_latex.append("\\midrule")

latex_out = (
    "\\begin{{table}}[htbp]\n\\centering\n"
    "\\caption{{Training and test macro F1 comparison. F1 Gap = Train F1 - Test F1; "
    "values near zero confirm consistent generalisation.}}\n"
    "\\label{{tab:train_test}}\n"
    "\\begin{{tabular}}{{llrrr}}\n\\toprule\n"
    "Model & Condition & Train F1 & Test F1 & Gap \\\\\n\\midrule\n"
    + "\n".join(rows_latex)
    + "\n\\bottomrule\n\\end{{tabular}}\n\\end{{table}}"
)
with open('table2_train_test.tex', 'w') as f:
    f.write(latex_out)
print("\ntable2_train_test.tex saved.")

# Bar chart
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
fig.suptitle('Training vs. Test Macro F1 Score Comparison', fontsize=13)
for ax, cond, cmap in zip(axes, ['Unbalanced', 'SMOTE'], ['viridis', 'plasma']):
    sub = (comparison_df[comparison_df['Condition'] == cond]
           [['Model', 'Train_F1', 'Test_F1']].set_index('Model'))
    sub.plot(kind='bar', ax=ax, colormap=cmap)
    ax.set_title(cond)
    ax.set_ylabel('Macro F1')
    ax.tick_params(axis='x', rotation=45)
    ax.legend(title='Metric')
    ax.grid(axis='y', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.savefig('train_test_comparison.pdf', dpi=300, bbox_inches='tight')
plt.show()

## Cell 13 -- McNemar's Test (All Pairwise, All 4 Classifiers, Both Conditions)

In [ ]:
# McNemar uses TEST SET predictions -- Stream 1 evaluation
# Test set is NEVER touched during training or cross-validation
alpha = 0.05

all_preds = {
    'LR (Unbal)':   pred_lr,
    'RF (Unbal)':   pred_rf,
    'SVM (Unbal)':  pred_svm,
    'XGB (Unbal)':  pred_xgb,
    'LR (SMOTE)':   pred_lr_bal,
    'RF (SMOTE)':   pred_rf_bal,
    'SVM (SMOTE)':  pred_svm_bal,
    'XGB (SMOTE)':  pred_xgb_bal,
}

y_arr = y_test.values
mcn_rows = []

for (na, pa), (nb, pb) in itertools.combinations(all_preds.items(), 2):
    ca = (pa == y_arr)
    cb = (pb == y_arr)
    tbl = [
        [int((ca & cb).sum()),  int((ca & ~cb).sum())],
        [int((~ca & cb).sum()), int((~ca & ~cb).sum())]
    ]
    try:
        res = mcnemar(tbl, exact=False)
        stat, pval = res.statistic, res.pvalue
        concl = "Reject H0 *" if pval < alpha else "Fail to reject H0"
    except Exception:
        stat, pval, concl = float('nan'), float('nan'), "N/A"

    mcn_rows.append({
        'Classifier A': na,
        'Classifier B': nb,
        'chi2(1)':      round(stat, 3),
        'p-value':      round(pval, 4),
        'Conclusion':   concl
    })

mcn_df = pd.DataFrame(mcn_rows)

print("=== McNemar's Test -- All Pairwise Results ===")
print(mcn_df.to_string(index=False))

# Key within-condition pairs (Experiment A)
print()
print("=== Key pairs: within Experiment A (Unbalanced condition) ===")
exp_a_models = {'LR (Unbal)', 'RF (Unbal)', 'SVM (Unbal)', 'XGB (Unbal)'}
key = mcn_df[
    mcn_df['Classifier A'].isin(exp_a_models) &
    mcn_df['Classifier B'].isin(exp_a_models)
]
print(key.to_string(index=False))

## Cell 14 -- 10-Fold CV on Training Partition + Cohen's d + Bootstrap CIs

**Reviewer 1 clarification (dual evaluation streams):**
- The 80/20 test split is fixed first. The test set is *never* seen during CV.
- 10-fold CV here runs exclusively on the 80% training partition with optimal hyperparameters fixed.
- SVM uses `probability=False` during CV (f1_macro does not need probabilities) -- this avoids
  the Platt scaling overhead that was causing runtime crashes.
- This produces Stream 2 evaluation, complementary to Stream 1 (held-out McNemar's test).

In [ ]:
# Verify train + test are non-overlapping
assert X_train_scaled.shape[0] + X_test_scaled.shape[0] == len(df), "Split size mismatch!"
print(f"Split verified: {X_train_scaled.shape[0]} train + {X_test_scaled.shape[0]} test = {len(df)}")
print("Test set is LOCKED -- not involved in any cross-validation below.")
print()

# 10-fold CV on TRAINING PARTITION ONLY
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=SEED)

# SVM CV: probability=False avoids Platt scaling overhead
# (f1_macro only needs class predictions, not probabilities)
clf_svm_cv = SVC(**best_svm_params, random_state=SEED,
                  probability=False, cache_size=2000)
clf_xgb_cv = XGBClassifier(**best_xgb_params, random_state=SEED,
                             eval_metric='logloss', use_label_encoder=False,
                             verbosity=0)

print("Running 10-fold CV on training partition...")
cv_scores = {}

cv_scores['Logistic Regression'] = cross_val_score(
    best_lr, X_train_scaled, y_train,
    scoring='f1_macro', cv=skf, n_jobs=-1
)
cv_scores['Random Forest'] = cross_val_score(
    best_rf, X_train_scaled, y_train,
    scoring='f1_macro', cv=skf, n_jobs=-1
)
cv_scores['SVM'] = cross_val_score(
    clf_svm_cv, X_train_scaled, y_train,
    scoring='f1_macro', cv=skf, n_jobs=-1
)
cv_scores['XGBoost'] = cross_val_score(
    clf_xgb_cv, X_train_scaled, y_train_xgb,
    scoring='f1_macro', cv=skf, n_jobs=-1
)

print()
print("10-fold CV results (training partition only):")
for name, sc in cv_scores.items():
    print(f"  {name}: mean={np.mean(sc):.4f}  std={np.std(sc):.5f}")

# Pairwise Cohen's d and Bootstrap CI
clf_names = list(cv_scores.keys())
eff_rows = []

for i in range(len(clf_names)):
    for j in range(i + 1, len(clf_names)):
        na = clf_names[i]
        nb = clf_names[j]
        sa = cv_scores[na]
        sb = cv_scores[nb]
        d = cohens_d(sa, sb)
        lo, hi = bootstrap_ci(sa, sb)
        near_zero_var = min(np.std(sa), np.std(sb)) < 1e-6
        eff_rows.append({
            'Classifier A':  na,
            'Classifier B':  nb,
            'Mean F1 Diff':  round(float(np.mean(sa) - np.mean(sb)), 4),
            "Cohen's d":     round(float(d), 3) if not (np.isinf(d) or np.isnan(d)) else str(d),
            'BS 95% CI':     f"[{lo:.4f}, {hi:.4f}]",
            'Note':          "Near-zero var -- use BS CI" if near_zero_var else ""
        })

eff_df = pd.DataFrame(eff_rows)
print()
print("=== Effect Size Analysis (Cohen's d + Bootstrap 95% CI) ===")
print(eff_df.to_string(index=False))
print()
print("Interpretation: |d|=0.2 small, |d|=0.5 medium, |d|=0.8 large")
print("When SVM CV variance is near-zero, Bootstrap CI is the primary summary.")

## Cell 15 -- Framework Overview Figure (Requested by Both Reviewers)

In [ ]:
# Colors
C_IN   = '#D6EAF8'  # Light blue
C_PROC = '#F2F3F4'  # Light grey
C_A    = '#D4EDDA'  # Light green (Track A)
C_B    = '#FDEBD0'  # Light orange (Track B)
C_EVAL = '#E8DAEF'  # Light purple
C_OUT  = 'white'

BH = 0.055
FS = 7
LW = 0.8

def box(ax, cx, cy, w, h, txt, fc, ec='black', ls='-', tc='black', fs=FS):
    ax.add_patch(patches.Rectangle(
        (cx - w / 2, cy - h / 2), w, h,
        facecolor=fc, edgecolor=ec, linestyle=ls, lw=LW,
        transform=ax.transAxes
    ))
    ax.text(cx, cy, txt, ha='center', va='center',
            fontsize=fs, color=tc, transform=ax.transAxes,
            multialignment='center')

def arr(ax, s, e, cs="arc3,rad=0"):
    ax.add_patch(patches.FancyArrowPatch(
        s, e, connectionstyle=cs, arrowstyle='-|>',
        color='#333333', lw=LW, mutation_scale=6,
        transform=ax.transAxes
    ))

fig, ax = plt.subplots(figsize=(7, 11))
ax.set_xlim(0, 1)
ax.set_ylim(-0.12, 1.02)
ax.axis('off')

# Y levels
y0  = 0.96; y1  = 0.86; y2  = 0.76; y3  = 0.65
y4  = 0.55; y5  = 0.44; y6  = 0.33; y7  = 0.21
y8  = 0.10; y9  = 0.00; y10 = -0.09
xl  = 0.27; xr  = 0.73; xc  = 0.5

box(ax, xc, y0,  0.33, BH,      "Input: UCI Phishing Websites Dataset\n(11,055 instances, 30 features)", C_IN)
box(ax, xc, y1,  0.22, BH,      "Stratified 80/20 Train-Test Split", C_PROC)
box(ax, xl, y2,  0.22, BH,      "Training Set\n(8,844 instances)", C_IN)
box(ax, xr, y2,  0.22, BH,      "Test Set (2,211)\n[LOCKED -- never in CV]", C_IN,
    ec='red', tc='red')
box(ax, xl, y3,  0.24, BH,      "Track A: Original Training\n(Imbalanced)", C_A)
box(ax, xr, y3,  0.24, BH,      "Track B: SMOTE-Balanced\nTraining", C_B)
box(ax, xc, y4,  0.40, BH * 1.2,"StandardScaler (fit on train only)\n"
    "GridSearchCV / RandomizedSearchCV (5-fold, f1_macro)\n"
    "for LR, RF, SVM (no prob during search), XGBoost", C_PROC)
box(ax, xc, y5,  0.40, BH,      "Best-parameter models refitted on full training data\n"
    "(SVM: probability=True added only here, once)", C_PROC)
box(ax, xl, y6,  0.24, BH * 1.3,"Stream 1 -- Held-out Test Set\n"
    "Classification metrics (Table 1)\n"
    "McNemar's test (pairwise)", C_EVAL)
box(ax, xr, y6,  0.24, BH * 1.3,"Stream 2 -- 10-fold CV on Train Only\n"
    "Cohen's d + Bootstrap 95% CIs\n"
    "SVM: probability=False for speed", C_EVAL)
box(ax, xc, y7,  0.40, BH,      "SHAP Interpretability (TreeExplainer: RF/XGB,\n"
    "LinearExplainer: LR, KernelExplainer: SVM)", C_PROC)
box(ax, xc, y8,  0.36, BH,      "Error Overlap Analysis\n"
    "(shared failure modes across classifier families)", C_PROC)
box(ax, xl, y9,  0.22, BH * 0.8,"Tables 1 & 2", C_OUT)
box(ax, xr, y9,  0.22, BH * 0.8,"Figures 1-8", C_OUT)

# Arrows
arr(ax, (xc, y0 - BH / 2), (xc, y1 + BH / 2))
arr(ax, (xc, y1 - BH / 2), (xl, y2 + BH / 2), "arc3,rad=0.15")
arr(ax, (xc, y1 - BH / 2), (xr, y2 + BH / 2), "arc3,rad=-0.15")
arr(ax, (xl, y2 - BH / 2), (xl, y3 + BH / 2))
arr(ax, (xl, y2 - BH / 2), (xr, y3 + BH / 2), "arc3,rad=-0.3")
arr(ax, (xl, y3 - BH / 2), (xc, y4 + BH * 0.6), "arc3,rad=0.15")
arr(ax, (xr, y3 - BH / 2), (xc, y4 + BH * 0.6), "arc3,rad=-0.15")
arr(ax, (xc, y4 - BH * 0.6), (xc, y5 + BH / 2))
arr(ax, (xc, y5 - BH / 2), (xl, y6 + BH * 0.65), "arc3,rad=0.2")
arr(ax, (xc, y5 - BH / 2), (xr, y6 + BH * 0.65), "arc3,rad=-0.2")
arr(ax, (xl, y6 - BH * 0.65), (xc, y7 + BH / 2), "arc3,rad=0.2")
arr(ax, (xr, y6 - BH * 0.65), (xc, y7 + BH / 2), "arc3,rad=-0.2")
arr(ax, (xc, y7 - BH / 2), (xc, y8 + BH / 2))
arr(ax, (xc, y8 - BH / 2), (xl, y9 + BH * 0.4), "arc3,rad=0.15")
arr(ax, (xc, y8 - BH / 2), (xr, y9 + BH * 0.4), "arc3,rad=-0.15")

ax.text(xc, 1.005, "Significance-Aware Benchmarking Pipeline",
        ha='center', va='bottom', fontsize=9,
        fontweight='bold', transform=ax.transAxes)

plt.tight_layout()
plt.savefig('framework_figure.pdf', dpi=300, bbox_inches='tight')
plt.show()
print("framework_figure.pdf saved.")

## Cell 16 -- Consolidated Hyperparameter Table (LaTeX)

In [ ]:
def fmt_range(lst):
    return '{' + ', '.join(str(x) for x in lst) + '}'


def flag_boundary(val, lst):
    """Add dagger if value is at min or max of numeric search range."""
    try:
        nums = [x for x in lst if isinstance(x, (int, float))]
        if nums and val in [min(nums), max(nums)]:
            return str(val) + 'dagger'
    except Exception:
        pass
    if isinstance(lst, list) and len(lst) > 1:
        if val == lst[0] or val == lst[-1]:
            return str(val) + 'dagger'
    return str(val)


rows = []

# Logistic Regression
lr_C_range = [0.001, 0.01, 0.1, 1, 10, 100]
lp  = best_lr.get_params()
lpb = best_lr_bal.get_params()
for param, grid in [
    ('C',        lr_C_range),
    ('penalty',  ['l1', 'l2', 'elasticnet']),
    ('solver',   ['liblinear', 'saga', 'newton-cg', 'lbfgs', 'sag']),
    ('l1_ratio', [0.1, 0.5, 0.9])
]:
    rows.append({
        'Classifier':       'Logistic Regression',
        'Hyperparameter':   param,
        'Search Range':     fmt_range(grid),
        'Optimal (Unbal)':  flag_boundary(lp.get(param, 'N/A'), grid),
        'Optimal (SMOTE)':  flag_boundary(lpb.get(param, 'N/A'), grid)
    })

# Random Forest
rf_grid = {
    'n_estimators':      [100, 200],
    'max_depth':         [10, 20, 30],
    'min_samples_split': [2, 5]
}
rp  = best_rf.get_params()
rpb = best_rf_bal.get_params()
for param, grid in rf_grid.items():
    rows.append({
        'Classifier':      'Random Forest',
        'Hyperparameter':  param,
        'Search Range':    fmt_range(grid),
        'Optimal (Unbal)': flag_boundary(rp.get(param), grid),
        'Optimal (SMOTE)': flag_boundary(rpb.get(param), grid)
    })

# SVM
svm_C_range = [0.1, 1, 10, 100, 1000]
svm_g_range = ['scale', 'auto', 0.001, 0.01, 0.1]
for param, grid in [
    ('C',      svm_C_range),
    ('kernel', ['linear', 'rbf']),
    ('gamma',  svm_g_range)
]:
    rows.append({
        'Classifier':      'SVM',
        'Hyperparameter':  param,
        'Search Range':    fmt_range(grid),
        'Optimal (Unbal)': flag_boundary(best_svm_params.get(param, 'N/A'), grid),
        'Optimal (SMOTE)': flag_boundary(best_svm_bal_params.get(param, 'N/A'), grid)
    })

# XGBoost
xgb_grid = {
    'n_estimators':     [100, 200, 300],
    'max_depth':        [3, 6, 9],
    'learning_rate':    [0.05, 0.1, 0.2],
    'subsample':        [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}
for param, grid in xgb_grid.items():
    rows.append({
        'Classifier':      'XGBoost',
        'Hyperparameter':  param,
        'Search Range':    fmt_range(grid),
        'Optimal (Unbal)': flag_boundary(best_xgb_params.get(param), grid),
        'Optimal (SMOTE)': flag_boundary(best_xgb_bal_params.get(param), grid)
    })

hp_df = pd.DataFrame(rows)

# Replace 'dagger' with actual dagger symbol for LaTeX
hp_df_latex = hp_df.copy()
for col in ['Optimal (Unbal)', 'Optimal (SMOTE)']:
    hp_df_latex[col] = hp_df_latex[col].str.replace('dagger', r'$\dagger$', regex=False)

print(hp_df.to_string(index=False))

latex_tbl = hp_df_latex.to_latex(
    index=False, escape=False, column_format='llccc',
    caption=(
        "Hyperparameter search ranges and optimal values for all classifiers "
        "under both experimental conditions. "
        "$\\dagger$ indicates a value at the grid boundary."
    ),
    label="tab:hyperparams"
)
with open('hyperparameter_table.tex', 'w') as f:
    f.write(latex_tbl)
print("\nhyperparameter_table.tex saved. (dagger marks boundary values)")

## Cell 17 -- ROC Curves (All 4 Classifiers)

In [ ]:
from sklearn.metrics import roc_curve, auc as sk_auc

fig, ax = plt.subplots(figsize=(7, 5))
line_styles = [('-', 2.0), ('--', 2.0), ('-.', 2.0), (':', 2.5)]

for (name, model), (ls, lw) in zip(
    [
        ('Logistic Regression', best_lr),
        ('Random Forest',       best_rf),
        ('SVM',                 best_svm),
        ('XGBoost',             best_xgb)
    ],
    line_styles
):
    if name == 'XGBoost':
        y_prob = model.predict_proba(X_test_scaled)[:, 1]
        fpr, tpr, _ = roc_curve(y_test_xgb, y_prob)
        roc_auc = sk_auc(fpr, tpr)
        ax.plot(fpr, tpr, ls=ls, lw=lw, alpha=0.9,
                label=f"{name} (AUC = {roc_auc:.4f})")
    else:
        RocCurveDisplay.from_estimator(
            model, X_test_scaled, y_test,
            name=name, ax=ax, ls=ls, lw=lw, alpha=0.9, pos_label=1
        )

ax.plot([0, 1], [0, 1], 'k--', lw=0.8, label='Random Classifier (AUC = 0.50)')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves for Classification Models')
ax.legend(loc='lower right', fontsize=8)
plt.tight_layout()
plt.savefig('fig_roc_curves.pdf', dpi=300, bbox_inches='tight')
plt.show()

## Cell 18 -- Confusion Matrices (4 Classifiers, Unbalanced Condition)

In [ ]:
models_cm = [
    ('Logistic Regression', pred_lr),
    ('Random Forest',       pred_rf),
    ('SVM',                 pred_svm),
    ('XGBoost',             pred_xgb),
]

y_arr = y_test.values
fig, axes = plt.subplots(1, 4, figsize=(22, 5))

for ax, (name, preds) in zip(axes, models_cm):
    cm = confusion_matrix(y_arr, preds, labels=[-1, 1])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Legitimate', 'Phishing'],
                yticklabels=['Legitimate', 'Phishing'])
    fn = cm[1, 0]
    fp = cm[0, 1]
    ax.set_title(f"{name}\nFP={fp}  FN={fn}", fontsize=8)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

plt.suptitle('Confusion Matrices -- Experiment A (Unbalanced Training)', y=1.01)
plt.tight_layout()
plt.savefig('fig_confusion_matrices.pdf', dpi=300, bbox_inches='tight')
plt.show()

print("False Negative counts (phishing missed as legitimate):")
for name, preds in models_cm:
    cm = confusion_matrix(y_arr, preds, labels=[-1, 1])
    print(f"  {name}: FN={cm[1, 0]}  FP={cm[0, 1]}")

## Cell 19 -- Gini Feature Importance (Random Forest)

In [ ]:
fi = pd.Series(best_rf.feature_importances_, index=feature_names).nlargest(15)

plt.figure(figsize=(10, 7))
sns.barplot(x=fi.values, y=fi.index, palette='viridis', orient='h')
plt.xlabel('Feature Importance (Gini Impurity Decrease)')
plt.title('Top 15 Feature Importances from Random Forest Classifier')
plt.tight_layout()
plt.savefig('fig_gini_importance.pdf', dpi=300, bbox_inches='tight')
plt.show()

print("Top 5 features:")
for feat, imp in fi.head(5).items():
    print(f"  {feat}: {imp:.4f}")

## Cell 20 -- SHAP Analysis (All 4 Classifiers)

- **RF / XGBoost**: TreeExplainer (exact, polynomial time)
- **Logistic Regression**: LinearExplainer (exact under linear assumption)
- **SVM**: KernelExplainer with 50 k-means background centroids and 200-instance test sample (approx 2-3 min)

In [ ]:
import shap
import warnings
warnings.filterwarnings('ignore')
shap.initjs()

# ---- RF: TreeExplainer ----
explainer_rf  = shap.TreeExplainer(best_rf)
sv_rf_raw     = explainer_rf.shap_values(X_test_scaled)

if isinstance(sv_rf_raw, list):
    sv_rf = sv_rf_raw[1]
elif sv_rf_raw.ndim == 3:
    sv_rf = sv_rf_raw[:, :, 1]
else:
    sv_rf = sv_rf_raw

# ---- XGBoost: TreeExplainer ----
explainer_xgb = shap.TreeExplainer(best_xgb)
sv_xgb_raw    = explainer_xgb.shap_values(X_test_scaled)

if isinstance(sv_xgb_raw, list):
    sv_xgb = sv_xgb_raw[1]
elif sv_xgb_raw.ndim == 3:
    sv_xgb = sv_xgb_raw[:, :, 1]
else:
    sv_xgb = sv_xgb_raw

# ---- Logistic Regression: LinearExplainer ----
bg_lr         = shap.maskers.Independent(X_train_scaled, max_samples=500)
explainer_lr  = shap.LinearExplainer(best_lr, bg_lr)
sv_lr_raw     = explainer_lr.shap_values(X_test_scaled)
sv_lr         = sv_lr_raw[1] if isinstance(sv_lr_raw, list) else sv_lr_raw

# ---- SVM: KernelExplainer (~2-3 min) ----
print("Computing SVM SHAP values via KernelExplainer (approx 2-3 min)...")
bg_svm        = shap.kmeans(X_train_scaled, 50)
explainer_svm = shap.KernelExplainer(best_svm.predict_proba, bg_svm)
np.random.seed(SEED)
sample_idx    = np.random.choice(len(X_test_scaled), 200, replace=False)
X_test_sample = X_test_scaled[sample_idx]
sv_svm_raw    = explainer_svm.shap_values(X_test_sample)

if isinstance(sv_svm_raw, list):
    sv_svm = sv_svm_raw[1]
elif sv_svm_raw.ndim == 3:
    sv_svm = sv_svm_raw[:, :, 1]
else:
    sv_svm = sv_svm_raw

print("SHAP computation complete.")
print(f"  RF:  {sv_rf.shape}  XGB: {sv_xgb.shape}  LR: {sv_lr.shape}  SVM: {sv_svm.shape}")

# Mean absolute SHAP per feature
def mean_abs(sv, names):
    return pd.Series(np.abs(sv).mean(axis=0), index=names)

shap_rf  = mean_abs(sv_rf,  feature_names)
shap_xgb = mean_abs(sv_xgb, feature_names)
shap_lr  = mean_abs(sv_lr,  feature_names)
shap_svm = mean_abs(sv_svm, feature_names)

top15 = shap_rf.nlargest(15).index.tolist()
shap_comparison = pd.DataFrame({
    'Random Forest': shap_rf[top15],
    'XGBoost':       shap_xgb[top15],
    'Logistic Reg':  shap_lr[top15],
    'SVM':           shap_svm[top15]
}, index=top15)

# Spearman rank correlations (cross-model feature-ranking agreement)
print()
print("=== Spearman Rank Correlation -- Cross-model SHAP Feature Ranking ===")
model_pairs = [
    ('RF',  shap_rf),
    ('XGB', shap_xgb),
    ('LR',  shap_lr),
    ('SVM', shap_svm)
]
for i, (na, sa) in enumerate(model_pairs):
    for nb, sb in model_pairs[i + 1:]:
        rho, pval = spearmanr(sa, sb)
        print(f"  {na} vs {nb}: rho={rho:.3f}  p={pval:.4f}")

## Cell 21 -- Figure: SHAP Beeswarm Summary (Random Forest)

In [ ]:
plt.figure(figsize=(10, 7))
shap.summary_plot(
    sv_rf, X_test_scaled,
    feature_names=feature_names,
    max_display=15, show=False, plot_type='dot'
)
plt.title('SHAP Summary -- Random Forest (Phishing Class)', fontsize=12)
plt.tight_layout()
plt.savefig('fig_shap_beeswarm_rf.pdf', dpi=300, bbox_inches='tight')
plt.show()

## Cell 22 -- Figure: Mean |SHAP| Comparison (All 4 Classifiers)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
shap_comparison.plot(kind='barh', ax=ax, width=0.8, alpha=0.9)
ax.invert_yaxis()
ax.set_xlabel('Mean |SHAP Value|', fontsize=11)
ax.set_title(
    'Model-Agnostic Feature Importance (Mean |SHAP|)\nTop 15 Features -- All Classifiers',
    fontsize=12
)
ax.legend(title='Classifier', loc='lower right')
plt.tight_layout()
plt.savefig('fig_shap_comparison.pdf', dpi=300, bbox_inches='tight')
plt.show()

## Cell 23 -- Figure: SHAP Dependence Plot (SSLfinal_State x URL_of_Anchor)

In [ ]:
ssl_idx    = feature_names.index('SSLfinal_State')
anchor_idx = feature_names.index('URL_of_Anchor')

plt.figure(figsize=(8, 5))
shap.dependence_plot(
    ssl_idx, sv_rf, X_test_scaled,
    feature_names=feature_names,
    interaction_index=anchor_idx,
    show=False, ax=plt.gca()
)
plt.title(
    'SHAP Dependence Plot -- SSLfinal_State (RF, interaction: URL_of_Anchor)',
    fontsize=11
)
plt.tight_layout()
plt.savefig('fig_shap_dependence.pdf', dpi=300, bbox_inches='tight')
plt.show()

## Cell 24 -- Error Overlap Analysis (Classifier Disagreement Patterns)

In [ ]:
y_arr = y_test.values
c_lr  = (pred_lr  == y_arr)
c_rf  = (pred_rf  == y_arr)
c_svm = (pred_svm == y_arr)
c_xgb = (pred_xgb == y_arr)

error_counts = {
    'LR only':   int((~c_lr  &  c_rf  &  c_svm &  c_xgb).sum()),
    'RF only':   int(( c_lr  & ~c_rf  &  c_svm &  c_xgb).sum()),
    'SVM only':  int(( c_lr  &  c_rf  & ~c_svm &  c_xgb).sum()),
    'XGB only':  int(( c_lr  &  c_rf  &  c_svm & ~c_xgb).sum()),
    'RF+SVM':    int(( c_lr  & ~c_rf  & ~c_svm &  c_xgb).sum()),
    'All wrong': int((~c_lr  & ~c_rf  & ~c_svm & ~c_xgb).sum()),
}

print("Error overlap counts:")
for k, v in error_counts.items():
    print(f"  {k}: {v}")

fig, ax = plt.subplots(figsize=(9, 5))
colors = ['#E74C3C', '#2980B9', '#27AE60', '#F39C12', '#8E44AD', '#95A5A6']
bars   = ax.bar(error_counts.keys(), error_counts.values(), color=colors)
ax.bar_label(bars, padding=3)
ax.set_ylabel('Number of Instances')
ax.set_title('Classifier Disagreement Patterns on Test Set (n=2,211)')
plt.tight_layout()
plt.savefig('fig_error_overlap.pdf', dpi=300, bbox_inches='tight')
plt.show()

# Hard-case feature profile: phishing instances that RF and SVM both miss
hard_mask = (~c_rf) & (~c_svm) & (y_arr == 1)
print(f"\nShared RF+SVM false negatives: {hard_mask.sum()} instances")
if hard_mask.sum() > 0:
    hard_X  = pd.DataFrame(X_test_scaled[hard_mask],  columns=feature_names)
    all_ph  = pd.DataFrame(X_test_scaled[y_arr == 1], columns=feature_names)
    diff    = (hard_X.mean() - all_ph.mean()).abs().nlargest(5)
    print("Top 5 features distinguishing hard cases from typical phishing instances:")
    print(diff.to_string())
    print("(Near-intermediate values in SSLfinal_State and URL_of_Anchor explain shared failures.)")